# 07 Job Storage and IT Market Statistics

This notebook creates IT market statistics using two approaches.

Part A:
- Main system statistics based on LLM-structured job data stored in MongoDB.
- These statistics are calculated from fields such as required_skills, programming_languages, frameworks, databases, cloud tools, job_category and seniority level.

Part B:
- Optional keyword-based market overview from the cleaned IT job postings dataset.
- This part is used only as a broader supporting analysis and not as the main statistics of the system.

The main statistics of the system are based on structured information extracted by the LLM.

## 1. Imports and Settings

In [1]:
import os
import json
import re
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
from collections import Counter

In [2]:
PROCESSED_JOBS_DIR = Path("../data/processed/jobs")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
DATABASE_DIR = Path("../database")
MARKET_STATS_DIR = Path("../outputs/market_statistics")

In [3]:
cleaned_jobs_path = PROCESSED_JOBS_DIR / "cleaned_it_job_postings.csv"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"
database_path = DATABASE_DIR / "jobs.sqlite"

In [4]:
load_dotenv(dotenv_path=Path("../.env"))
MONGODB_URI = os.getenv("MONGODB_URI")

In [5]:
client = MongoClient(MONGODB_URI)
db = client["cv_job_matching_agent"]
jobs_collection = db["analyzed_jobs"]

## Part A: LLM-Based Statistics from MongoDB

## 2. Load Processed Job Postings from MongoDB

In [6]:
job_documents = list(jobs_collection.find({}))

## 3. Define Helper Functions for Counting Statistics

In [7]:
def add_list_items_to_counter(counter, items, weight=1):
    if not isinstance(items, list):
        return

    for item in items:
        if item is None:
            continue

        item_clean = str(item).strip()

        if item_clean == "" or item_clean.lower() in ["none", "null", "nan", "unknown"]:
            continue

        counter[item_clean] += weight

In [8]:
def add_value_to_counter(counter, value, weight=1):
    """
    Adds one categorical value to a Counter.
    """
    if value is None:
        value = "Unknown"

    value_clean = str(value).strip()

    if value_clean == "" or value_clean.lower() in ["none", "null", "nan"]:
        value_clean = "Unknown"

    counter[value_clean] += weight

In [9]:
def counter_to_dataframe(counter, value_column_name):
    """
    Converts Counter result to a DataFrame for display and CSV export.
    """
    return pd.DataFrame(
        counter.most_common(),
        columns=[value_column_name, "count"]
    )

## 4. Create Counters for Structured Job Fields

In [10]:
job_category_counter = Counter()
seniority_counter = Counter()
work_mode_counter = Counter()
employment_type_counter = Counter()
minimum_years_counter = Counter()

required_skills_counter = Counter()
nice_to_have_skills_counter = Counter()
programming_languages_counter = Counter()
frameworks_counter = Counter()
databases_counter = Counter()
cloud_devops_counter = Counter()
data_ai_tools_counter = Counter()
testing_tools_counter = Counter()
soft_skills_counter = Counter()

monthly_unique_jobs_counter = Counter()
monthly_submission_counter = Counter()

In [11]:
len(job_documents)

2

## 5. Count Statistics from LLM-Structured Job Data

In [12]:
for doc in job_documents:
    structured_job = doc.get("structured_job", {}) or {}

    submission_count = doc.get("submission_count", 1)

    if submission_count is None:
        submission_count = 1

    submission_count = int(submission_count)

    add_value_to_counter(
        job_category_counter,
        structured_job.get("job_category"),
        submission_count
    )

    add_value_to_counter(
        work_mode_counter,
        structured_job.get("work_mode"),
        submission_count
    )

    add_value_to_counter(
        employment_type_counter,
        structured_job.get("employment_type"),
        submission_count
    )

    experience_requirements = structured_job.get("experience_requirements", [])

    if isinstance(experience_requirements, list) and len(experience_requirements) > 0:
        first_experience = experience_requirements[0]
        add_value_to_counter(
            seniority_counter,
            first_experience.get("seniority_level"),
            submission_count
        )
        add_value_to_counter(
            minimum_years_counter,
            first_experience.get("minimum_years"),
            submission_count
        )
    else:
        add_value_to_counter(
            seniority_counter,
            "Unknown",
            submission_count
        )

    add_list_items_to_counter(
        required_skills_counter,
        structured_job.get("required_skills", []),
        submission_count
    )

    add_list_items_to_counter(
        nice_to_have_skills_counter,
        structured_job.get("nice_to_have_skills", []),
        submission_count
    )

    add_list_items_to_counter(
        programming_languages_counter,
        structured_job.get("programming_languages", []),
        submission_count
    )

    add_list_items_to_counter(
        frameworks_counter,
        structured_job.get("frameworks_and_libraries", []),
        submission_count
    )

    add_list_items_to_counter(
        databases_counter,
        structured_job.get("databases", []),
        submission_count
    )

    add_list_items_to_counter(
        cloud_devops_counter,
        structured_job.get("cloud_and_devops_tools", []),
        submission_count
    )

    add_list_items_to_counter(
        data_ai_tools_counter,
        structured_job.get("data_and_ai_tools", []),
        submission_count
    )

    add_list_items_to_counter(
        testing_tools_counter,
        structured_job.get("testing_tools", []),
        submission_count
    )

    add_list_items_to_counter(
        soft_skills_counter,
        structured_job.get("soft_skills", []),
        submission_count
    )

## 6. Monthly Statistics from Processed Job Postings

In [13]:
for doc in job_documents:
    first_seen_at = doc.get("first_seen_at")
    submission_count = doc.get("submission_count", 1)

    if submission_count is None:
        submission_count = 1

    if first_seen_at is None:
        continue

    parsed_date = pd.to_datetime(first_seen_at, errors="coerce")

    if pd.isna(parsed_date):
        continue

    year_month = parsed_date.strftime("%Y-%m")
    
    monthly_unique_jobs_counter[year_month] += 1

    monthly_submission_counter[year_month] += int(submission_count)

## 7. Convert Counter Results to DataFrames

In [14]:
llm_job_category_stats = counter_to_dataframe(
    job_category_counter,
    "job_category"
)

llm_seniority_stats = counter_to_dataframe(
    seniority_counter,
    "seniority_level"
)

llm_work_mode_stats = counter_to_dataframe(
    work_mode_counter,
    "work_mode"
)

llm_employment_type_stats = counter_to_dataframe(
    employment_type_counter,
    "employment_type"
)

llm_minimum_years_stats = counter_to_dataframe(
    minimum_years_counter,
    "minimum_years"
)

llm_required_skills_stats = counter_to_dataframe(
    required_skills_counter,
    "skill"
)

llm_nice_to_have_skills_stats = counter_to_dataframe(
    nice_to_have_skills_counter,
    "skill"
)

llm_programming_languages_stats = counter_to_dataframe(
    programming_languages_counter,
    "programming_language"
)

llm_frameworks_stats = counter_to_dataframe(
    frameworks_counter,
    "framework"
)

llm_databases_stats = counter_to_dataframe(
    databases_counter,
    "database"
)

llm_cloud_devops_stats = counter_to_dataframe(
    cloud_devops_counter,
    "tool"
)

llm_data_ai_tools_stats = counter_to_dataframe(
    data_ai_tools_counter,
    "tool"
)

llm_testing_tools_stats = counter_to_dataframe(
    testing_tools_counter,
    "tool"
)

llm_soft_skills_stats = counter_to_dataframe(
    soft_skills_counter,
    "soft_skill"
)

llm_monthly_unique_jobs_stats = counter_to_dataframe(
    monthly_unique_jobs_counter,
    "year_month"
)

llm_monthly_submission_stats = counter_to_dataframe(
    monthly_submission_counter,
    "year_month"
)

## 8. Preview Main LLM-Based Statistics

In [15]:
llm_required_skills_stats.head(20)

,skill,count
0,JavaScript,2
1,HTML5,2
2,TypeScript,1
3,React,1
4,Node.js,1
5,Python,1
6,Express.js,1
7,PostgreSQL,1
8,MongoDB,1
9,CSS3,1


In [16]:
llm_programming_languages_stats.head(20)

,programming_language,count
0,JavaScript,2
1,TypeScript,1
2,Python,1
3,C#,1
4,VB.NET,1


In [17]:
llm_frameworks_stats.head(20)

,framework,count
0,React,1
1,Node.js,1
2,Express.js,1
3,ASP.NET,1
4,MVC,1
5,jQuery,1
6,Knockout.js,1
7,Angular.js,1
8,HTML5,1


In [18]:
llm_job_category_stats.head(20)

,job_category,count
0,Software Development,2


In [19]:
llm_seniority_stats.head(20)

,seniority_level,count
0,Unknown,1
1,Senior,1


In [20]:
llm_monthly_unique_jobs_stats.head(20)

,year_month,count
0,2026-07,2


In [21]:
llm_monthly_submission_stats.head(20)

,year_month,count
0,2026-07,2


## 9. Save LLM-Based Statistics

In [22]:
llm_job_category_stats.to_csv(
    MARKET_STATS_DIR / "llm_job_category_stats.csv",
    index=False
)

llm_seniority_stats.to_csv(
    MARKET_STATS_DIR / "llm_seniority_stats.csv",
    index=False
)

llm_work_mode_stats.to_csv(
    MARKET_STATS_DIR / "llm_work_mode_stats.csv",
    index=False
)

llm_employment_type_stats.to_csv(
    MARKET_STATS_DIR / "llm_employment_type_stats.csv",
    index=False
)

llm_minimum_years_stats.to_csv(
    MARKET_STATS_DIR / "llm_minimum_years_stats.csv",
    index=False
)

llm_required_skills_stats.to_csv(
    MARKET_STATS_DIR / "llm_required_skills_stats.csv",
    index=False
)

llm_nice_to_have_skills_stats.to_csv(
    MARKET_STATS_DIR / "llm_nice_to_have_skills_stats.csv",
    index=False
)

llm_programming_languages_stats.to_csv(
    MARKET_STATS_DIR / "llm_programming_languages_stats.csv",
    index=False
)

llm_frameworks_stats.to_csv(
    MARKET_STATS_DIR / "llm_frameworks_stats.csv",
    index=False
)

llm_databases_stats.to_csv(
    MARKET_STATS_DIR / "llm_databases_stats.csv",
    index=False
)

llm_cloud_devops_stats.to_csv(
    MARKET_STATS_DIR / "llm_cloud_devops_stats.csv",
    index=False
)

llm_data_ai_tools_stats.to_csv(
    MARKET_STATS_DIR / "llm_data_ai_tools_stats.csv",
    index=False
)

llm_testing_tools_stats.to_csv(
    MARKET_STATS_DIR / "llm_testing_tools_stats.csv",
    index=False
)

llm_soft_skills_stats.to_csv(
    MARKET_STATS_DIR / "llm_soft_skills_stats.csv",
    index=False
)

## 10. Save LLM-Based Statistics Summary

In [23]:
llm_statistics_summary = {
    "statistics_type": "LLM-based structured job statistics",
    "total_unique_processed_jobs": len(job_documents),
    "total_job_submissions": sum(
        int(doc.get("submission_count", 1) or 1)
        for doc in job_documents
    ),
    "top_job_categories": llm_job_category_stats.head(10).to_dict(orient="records"),
    "top_seniority_levels": llm_seniority_stats.head(10).to_dict(orient="records"),
    "top_required_skills": llm_required_skills_stats.head(10).to_dict(orient="records"),
    "top_programming_languages": llm_programming_languages_stats.head(10).to_dict(orient="records"),
    "top_frameworks": llm_frameworks_stats.head(10).to_dict(orient="records"),
    "top_databases": llm_databases_stats.head(10).to_dict(orient="records"),
    "top_cloud_devops_tools": llm_cloud_devops_stats.head(10).to_dict(orient="records"),
    "top_data_ai_tools": llm_data_ai_tools_stats.head(10).to_dict(orient="records"),
    "top_testing_tools": llm_testing_tools_stats.head(10).to_dict(orient="records"),
    "monthly_unique_jobs": llm_monthly_unique_jobs_stats.to_dict(orient="records"),
    "monthly_submissions": llm_monthly_submission_stats.to_dict(orient="records"),
}

In [24]:
llm_summary_path = MARKET_STATS_DIR / "llm_statistics_summary.json"

In [25]:
with open(llm_summary_path, "w", encoding="utf-8") as file:
    json.dump(llm_statistics_summary, file, indent=4, ensure_ascii=False)

## Part B: Optional Keyword-Based Market Overview

## 11. Load Cleaned IT Job Postings Dataset

In [26]:
df_jobs = pd.read_csv(cleaned_jobs_path)

## 12. Find Title and Description Columns

In [27]:
description_col = "clean_job_description"
title_col = "clean_job_title"

## 13. Create Market Text Column

In [28]:
df_jobs["market_text"] = (df_jobs[title_col].fillna("").astype(str) + " " + df_jobs[description_col].fillna("").astype(str)).str.lower()

## 14. Define Keyword Groups

In [29]:
keyword_groups = {
    "programming_languages": [
        "python", "java", "javascript", "typescript", "c#", "c++",
        "php", "go", "kotlin", "swift"
    ],
    "frameworks": [
        "react", "angular", "vue", "node.js", "django", "flask",
        "fastapi", "spring boot", ".net", "laravel"
    ],
    "databases": [
        "sql", "mysql", "postgresql", "mongodb", "oracle",
        "redis", "elasticsearch"
    ],
    "cloud_devops_tools": [
        "aws", "azure", "google cloud", "gcp", "docker",
        "kubernetes", "jenkins", "terraform", "linux"
    ],
    "data_ai_tools": [
        "power bi", "tableau", "pandas", "numpy", "spark",
        "tensorflow", "pytorch", "machine learning"
    ],
    "testing_tools": [
        "selenium", "cypress", "pytest", "junit", "postman", "jest"
    ]
}

## 15. Define Keyword Counting Function

In [30]:
def count_keywords_in_text(df, text_column, keywords):
    counter = Counter()

    if df.empty or text_column not in df.columns:
        return counter

    for keyword in keywords:
        count = df[text_column].str.contains(
            re.escape(keyword),
            case=False,
            na=False
        ).sum()

        counter[keyword] = int(count)

    return counter

## 16. Calculate Keyword-Based Statistics

In [31]:
keyword_statistics = {}

for group_name, keywords in keyword_groups.items():
    counter = count_keywords_in_text(
        df_jobs,
        "market_text",
        keywords
    )

    keyword_statistics[group_name] = counter_to_dataframe(
        counter,
        "keyword"
    )

In [32]:
keyword_statistics["programming_languages"].head(20)

,keyword,count
0,go,5276
1,python,2739
2,java,2393
3,javascript,1267
4,c#,681
5,c++,672
6,typescript,409
7,php,145
8,swift,138
9,kotlin,100


## 17. Save Keyword-Based Statistics

In [33]:
for group_name, stats_df in keyword_statistics.items():
    stats_df.to_csv(
        MARKET_STATS_DIR / f"keyword_{group_name}_stats.csv",
        index=False
    )

## 18. Save Combined Market Statistics Summary

In [34]:
combined_summary = {
    "main_statistics": {
        "type": "LLM-based structured statistics from MongoDB",
        "total_unique_processed_jobs": len(job_documents),
        "total_job_submissions": sum(
            int(doc.get("submission_count", 1) or 1)
            for doc in job_documents
        ),
        "top_required_skills": llm_required_skills_stats.head(10).to_dict(orient="records"),
        "top_programming_languages": llm_programming_languages_stats.head(10).to_dict(orient="records"),
        "top_frameworks": llm_frameworks_stats.head(10).to_dict(orient="records"),
        "top_job_categories": llm_job_category_stats.head(10).to_dict(orient="records")
    },
    "supporting_statistics": {
        "type": "Keyword-based overview from cleaned dataset",
        "total_cleaned_jobs": int(len(df_jobs)) if not df_jobs.empty else 0
    }
}

In [35]:
for group_name, stats_df in keyword_statistics.items():
    combined_summary["supporting_statistics"][f"top_{group_name}"] = (
        stats_df.head(10).to_dict(orient="records")
    )

In [36]:
combined_summary_path = MARKET_STATS_DIR / "combined_market_statistics_summary.json"

In [37]:
with open(combined_summary_path, "w", encoding="utf-8") as file:
    json.dump(combined_summary, file, indent=4, ensure_ascii=False)

## 19. Close MongoDB Connection

In [38]:
client.close()